## Track B: Multicultural Visual Reasoning

### 1. Environment Setup & OpenSearch Initialization

In [2]:
import os
import ast
import io
import base64
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Dataset
from openai import OpenAI
from PIL import Image

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

BASE_URL = os.getenv("BASE_URL") or "https://api.novasearch.org/gemma4/v1"
API_KEY = os.getenv("API_KEY") or "nova-vl"
MODEL = os.getenv("MODEL") or "google/gemma-4-31b-it"

# Define target Track B Indices
cvqa_index_name = f"{OPENSEARCH_USER}_cvqa_project"
wiki_cache_index = f"{OPENSEARCH_USER}_wiki_cache"

# Initialize OpenSearch Client
client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress=True, 
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl=True,
    url_prefix='opensearch_v3',
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)

# Initialize OpenAI server client for Gemma-4-31B with verified fallbacks
openai_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Initialize embedding models matching your Phase 2 vector fields
print("Loading embedding models...")
sbert_model = SentenceTransformer('all-mpnet-base-v2')       # 768 dim
bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')     # 384 dim
clip_model = SentenceTransformer('clip-ViT-B-32')             # 512 dim
print("Models loaded successfully.")

Loading embedding models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

0_CLIPModel/vocab.json: 0.00B [00:00, ?B/s]

0_CLIPModel/special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

0_CLIPModel/pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

0_CLIPModel/model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Models loaded successfully.


### 2. Dataset Loading and Stratified Held-out Split

In [ ]:
print("Loading afaji/cvqa dataset from Hugging Face...")
cvqa_ds = load_dataset("afaji/cvqa", split="test")

def parse_subset_metadata(example):
    try:
        # Extract Language and Country safely from the Subset tuple string
        subset_tuple = ast.literal_eval(example['Subset'])
        example['language'] = subset_tuple[0]
        example['country'] = subset_tuple[1]
    except:
        example['language'] = "Unknown"
        example['country'] = "Unknown"
    return example

# Map metadata and filter for target evaluation languages
cvqa_ds = cvqa_ds.map(parse_subset_metadata)
target_languages = ["English", "Portuguese", "Arabic"]
cvqa_filtered = cvqa_ds.filter(lambda x: x['language'] in target_languages)

# Convert to pandas to sample 1,000 rows proportionally
df_cvqa = cvqa_filtered.to_pandas()
sampled_df = df_cvqa.groupby('language', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 334), random_state=42)
)

# Convert back to Dataset and enforce exactly 1000 items
working_dataset = Dataset.from_pandas(sampled_df).shuffle(seed=42)
if len(working_dataset) > 1000:
    working_dataset = working_dataset.select(range(1000))

# Create stratified Train (Retrieval Corpus) and Held-Out Test Set
split_ds = working_dataset.train_test_split(test_size=0.2, stratify_by_column='language', seed=42)
retrieval_corpus = split_ds['train']
test_set = split_ds['test']

print(f"Dataset Split complete: Retrieval Split size = {len(retrieval_corpus)}, Blind Test Split size = {len(test_set)}")